<a href="https://colab.research.google.com/github/Ryan56-56/project1ML/blob/main/Another_copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import imageio
import torchvision
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score, f1_score

import matplotlib.pyplot as plt
import numpy as np

# Device
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", torch_device)


Using device: cuda


In [ ]:
##!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 86.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 60.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 90.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 64.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 115.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [ ]:
##!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
raw_data_train = '/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train'

dataset_train = []
labels_train  = []
targets_train = []

for folder in os.listdir(raw_data_train):
    for image in os.listdir(os.path.join(raw_data_train, folder)):
        if folder not in labels_train:
            labels_train.append(folder)

        targets_train.append(labels_train.index(folder))

        img_arr = imageio.imread(os.path.join(raw_data_train, folder, image), pilmode="RGB")
        img = torch.from_numpy(img_arr).permute(2, 0, 1).float() / 255
        dataset_train.append(img)

data_train = torch.stack(dataset_train)
targets_train = torch.tensor(targets_train, dtype=torch.long)

print("Train data shape:", data_train.shape)
print("Train labels shape:", targets_train.shape)


/tmp/ipykernel_11473/1927241243.py:14: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img_arr = imageio.imread(os.path.join(raw_data_train, folder, image), pilmode="RGB")


Train data shape: torch.Size([50001, 3, 32, 32])
Train labels shape: torch.Size([50001])


In [ ]:
raw_data_test = '/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test'

dataset_test = []
labels_test = []
targets_test = []

for folder in os.listdir(raw_data_test):
    for image in os.listdir(os.path.join(raw_data_test, folder)):
        if folder not in labels_test:
            labels_test.append(folder)

        targets_test.append(labels_test.index(folder))

        img_arr = imageio.imread(os.path.join(raw_data_test, folder, image), pilmode="RGB")
        img = torch.from_numpy(img_arr).permute(2, 0, 1).float() / 255
        dataset_test.append(img)

data_test = torch.stack(dataset_test)
targets_test = torch.tensor(targets_test, dtype=torch.long)

print("Test data shape:", data_test.shape)
print("Test labels shape:", targets_test.shape)


/tmp/ipykernel_11473/1158788841.py:14: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img_arr = imageio.imread(os.path.join(raw_data_test, folder, image), pilmode="RGB")


Test data shape: torch.Size([10000, 3, 32, 32])
Test labels shape: torch.Size([10000])


In [ ]:
X_train = data_train.float()
X_test  = data_test.float()

CIFAR_train_list = [(X_train[i], targets_train[i].item()) for i in range(X_train.shape[0])]
CIFAR_test_list  = [(X_test[i], targets_test[i].item()) for i in range(X_test.shape[0])]

train_dl = DataLoader(CIFAR_train_list, batch_size=16, shuffle=True)
test_dl  = DataLoader(CIFAR_test_list, batch_size=10000, shuffle=True)

print("Dataloaders ready")


Dataloaders ready


In [ ]:
class DL_3h_net(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(32*32*3, 200)
        self.linear2 = nn.Linear(200, 100)
        self.linear3 = nn.Linear(100, 50)
        self.linear4 = nn.Linear(50, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.linear1(x))
        x = torch.relu(self.linear2(x))
        x = torch.relu(self.linear3(x))
        return self.linear4(x)


class CNN_net(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 5, 1, 1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(16),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.2),

            nn.Conv2d(16, 32, 5, 1, 1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.2),

            nn.Conv2d(32, 32, 5, 1, 1),
            nn.LeakyReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.2),

            nn.Flatten(),
            nn.Linear(128, 512),
            nn.LeakyReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.model(x)


class MLP_net(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(32*32*3, 20)
        self.norm = nn.LayerNorm(20)
        self.dropout = nn.Dropout(0.2)
        self.linear2 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.leaky_relu(self.linear1(x))
        x = self.norm(x)
        x = self.dropout(x)
        return self.linear2(x)


In [ ]:
def training_loop(N_Epochs, model, loss_fn, opt, checkpoint_path):
    for epoch in range(N_Epochs):
        for xb, yb in train_dl:
            xb = xb.to(torch_device)
            yb = yb.to(torch_device)

            y_pred = model(xb)
            loss = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Loss = {loss.item():.4f}")
            # Save the model with a unique name per epoch
            torch.save(model.state_dict(), os.path.join(checkpoint_path, f"model_epoch_{epoch}.pth"))


In [ ]:
import os

CHECKPOINT_PATH = "/content/drive/MyDrive/CNN_model_CIFAR10"

# Create the directory if it doesn't exist
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

model = CNN_net().to(torch_device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

training_loop(60, model, loss_fn, opt, CHECKPOINT_PATH)

# Verify immediately after training if files were saved
print(f"Files in {CHECKPOINT_PATH} after training: {os.listdir(CHECKPOINT_PATH)}")

Epoch 0 | Loss = 2.4570
Epoch 5 | Loss = 2.8095
Epoch 10 | Loss = 2.4093
Epoch 15 | Loss = 3.1393
Epoch 20 | Loss = 1.6034
Epoch 25 | Loss = 0.9457
Epoch 30 | Loss = 1.2673
Epoch 35 | Loss = 1.0034
Epoch 40 | Loss = 1.7073
Epoch 45 | Loss = 1.9560
Epoch 50 | Loss = 1.0961
Epoch 55 | Loss = 3.1545
Files in /content/drive/MyDrive/CNN_model_CIFAR10 after training: ['model_epoch_0.pth', 'model_epoch_5.pth', 'model_epoch_10.pth', 'model_epoch_15.pth', 'model_epoch_20.pth', 'model_epoch_25.pth', 'model_epoch_30.pth', 'model_epoch_35.pth', 'model_epoch_40.pth', 'model_epoch_45.pth', 'model_epoch_50.pth', 'model_epoch_55.pth']


In [ ]:
import os
print(os.listdir(CHECKPOINT_PATH))

['model_epoch_0.pth', 'model_epoch_5.pth', 'model_epoch_10.pth', 'model_epoch_15.pth', 'model_epoch_20.pth', 'model_epoch_25.pth', 'model_epoch_30.pth', 'model_epoch_35.pth', 'model_epoch_40.pth', 'model_epoch_45.pth', 'model_epoch_50.pth', 'model_epoch_55.pth']


In [ ]:
def print_metrics_function(y_test, y_pred):
    print("Accuracy: %.2f" % accuracy_score(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Precision: %.3f" % precision_score(y_test, y_pred, average='weighted'))
    print("Recall: %.3f" % recall_score(y_test, y_pred, average='weighted'))
    print("F1-measure: %.3f" % f1_score(y_test, y_pred, average='weighted'))


In [ ]:
with torch.no_grad():
    for x_real, y_real in test_dl:
        x_real = x_real.to(torch_device)
        y_pred = model(x_real)
        _, preds = torch.max(y_pred, dim=1)
        print_metrics_function(y_real, preds.cpu())


Accuracy: 0.07
Confusion Matrix:
[[ 14  41   2 780   7  15  88   3  43   7]
 [754   8  56  24   8  37   3  56  21  33]
 [  4 811   4  31  13  10  28   5  75  19]
 [ 10   8  26  11 779  60   8  38  10  50]
 [ 59  12  45  10  41  49   3 676  35  70]
 [ 42  23 147  17  67 508  12  85  31  68]
 [ 31  14  65  11  44  56  10  88  69 612]
 [ 13  82   5  32   7  15  21  19 747  59]
 [  5  32   4  88   9   9 828   4  17   4]
 [ 58   4 608  11  31 166   8  47  11  56]]
Precision: 0.073
Recall: 0.069
F1-measure: 0.071


In [ ]:
model2 = CNN_net().to(torch_device)
# Corrected: Specify a specific checkpoint file to load, for example 'model_epoch_55.pth'
# You can choose any epoch for which a checkpoint was successfully saved.
# This cell will be executed again after successful training and file verification.
# For now, we are re-running training to ensure files are saved.
# model2.load_state_dict(torch.load(os.path.join(CHECKPOINT_PATH, "model_epoch_55.pth")))
# model2.eval()

# with torch.no_grad():
#     for x_real, y_real in test_dl:
#         x_real = x_real.to(torch_device)
#         y_pred = model2(x_real)
#         _, preds = torch.max(y_pred, dim=1)
#         print_metrics_function(y_real, preds.cpu())

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Untitled document (2).gdoc', 'Ryan.gslides', 'The Harlem Renaissance.gslides', 'Copy of Untitled document.gdoc', 'Valade, Ryan Final Essay.gdoc', 'Ryan Valade SOK Essay.gdoc', 'Copy of Ryan Valade SOK Essay.gdoc', 'Copy of Y5S3 ADS SENSITIVITY CALCULATOR.gsheet', 'Untitled document (1).gdoc', 'Ryan valade Research paper.gdoc', 'Untitled presentation.gslides', 'Untitled document.gdoc', ' data user_de 0 com.android.providers.telephony app_parts FILE_2279.pdf', ' data user_de 0 com.android.providers.telephony app_parts Blank (1).pdf', ' data user_de 0 com.android.providers.telephony app_parts Blank.pdf', 'Cooking Data', 'Colab Notebooks', 'EStatement-2025-05-23-65983.pdf', 'data.csv', 'CIFAR-10-images-master', 'CNN_model_CIFAR10', 'CNN_model_CIFAR100', 'CNN_model_CIFAR105', 'CNN_model_CIFAR1010', 'CNN_model_CIFAR1015', 'CNN_model_CIFAR1020', 'CNN_model_CIFAR1025', 'CNN_model_CIFAR1030', 'CNN_model_CIFAR1035', 'CNN_model_CIFAR1040', 'CNN_model_CIFAR1045', 'CNN_model_CIFAR1050', 'CNN_mode

In [ ]:
model_rc = nn.Sequential(
    nn.Conv2d(3, 16, 5, 1, 1),
    nn.LeakyReLU(),
    nn.BatchNorm2d(16),
    nn.MaxPool2d(2, 2),
    nn.Dropout(0.2),

    nn.Conv2d(16, 32, 5, 1, 1),
    nn.LeakyReLU(),
    nn.BatchNorm2d(32),
    nn.MaxPool2d(2, 2),
    nn.Dropout(0.2),

    nn.Conv2d(32, 32, 5, 1, 1),
    nn.LeakyReLU(),
    nn.BatchNorm2d(32),
    nn.MaxPool2d(2, 2),
    nn.Dropout(0.2),

    nn.Flatten()
)

test_tensor = torch.randn(16, 3, 32, 32)
res = model_rc(test_tensor)
print("CNN Flatten Output Shape:", res.shape)


CNN Flatten Output Shape: torch.Size([16, 128])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
